# Agrupamento e avaliação

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Igorbraziel/juristcu-clustering-tsdae/blob/main/notebooks/03_clustering.ipynb)

- **Objetivo:** agrupar os enunciados com k-means e testar H1 e H2.
- **Entradas:** representações salvas pelo `02_embeddings`; áreas e temas do JurisTCU.
- **Saídas:** tabelas em `results/tables/` e figuras em `results/figures/`.
- **Requisitos:** não requer GPU. Áreas e temas entram apenas na avaliação.

In [ ]:
# Configuração do ambiente. Fora do Colab, usa o ambiente local criado por `uv sync`.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # Trocar pelo nome de uma branch de trabalho para testá-la.
REPO_URL = "https://github.com/Igorbraziel/juristcu-clustering-tsdae.git"
REPO_DIR = "/content/juristcu-clustering-tsdae"


def run(command: str) -> None:
    """Executa um comando de shell e interrompe a célula em caso de erro."""
    subprocess.run(command, shell=True, check=True)


if IN_COLAB:
    # Clona o repositório, se necessário, e posiciona na versão remota da branch.
    run(f"test -d {REPO_DIR} || git clone -q {REPO_URL} {REPO_DIR}")
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
    run(f"git -C {REPO_DIR} checkout -q -B {BRANCH} FETCH_HEAD")
    # O uv costuma vir no Colab. Instala apenas se faltar.
    run("command -v uv > /dev/null || pip install -q uv")
    # --system: instala no Python do Colab, que já traz o torch com CUDA.
    # --no-sources: ignora o índice de torch CPU do pyproject.toml.
    # --no-config: ignora configurações locais do uv.
    # Instalação não editável: disponível sem reiniciar o ambiente de execução.
    # Nunca usar `uv sync` nem requirements.txt no Colab.
    run(f"uv pip install --system --no-config --no-sources -q {REPO_DIR}")
    # Tabelas e figuras são gravadas em results/, a partir da raiz do repositório.
    os.chdir(REPO_DIR)
    # Após atualizar o repositório com mudanças em src/, reiniciar o ambiente de execução.
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import importlib

import pandas as pd

# Atualiza o cache de importação para reconhecer o pacote recém-instalado.
importlib.invalidate_caches()

from juristcu import analysis, clustering, data, evaluation, storage  # noqa: E402
from juristcu.config import (  # noqa: E402
    AREA_COLUMN,
    K_RANGE,
    MIN_TOPIC_SIZE,
    N_BOOTSTRAP,
    SEED,
    TEXT_COLUMN,
    TOPIC_COLUMN,
    TSDAE_SEEDS,
)

storage.save_environment("03")

## 1. Carregamento

- Representações salvas pelo `02_embeddings`, na ordem dos documentos.
- R3 aparece uma vez por semente do TSDAE.

In [ ]:
documents = data.load_documents()
texts = documents[TEXT_COLUMN].tolist()
areas = documents[AREA_COLUMN].to_numpy()
topics = documents[TOPIC_COLUMN].to_numpy()
n_areas = documents[AREA_COLUMN].nunique()

r3_names = [f"R3-seed{seed}" for seed in TSDAE_SEEDS]
names = ["R1", "R2", "R2-CLS", *r3_names, "R4"]
representations = {name: storage.load_embeddings(name) for name in names}
n_areas

## 2. k-means por área

- k igual ao número de áreas, para todas as representações.
- Mesma semente para todas as representações.

In [ ]:
area_preds = {
    name: clustering.run_kmeans(x, k=n_areas, seed=SEED) for name, x in representations.items()
}

## 3. Métricas por área

- NMI (principal) e ARI em relação às áreas. Silhueta como medida interna.
- IC de 95% da NMI por bootstrap dos documentos.

In [ ]:
area_cis = {
    name: evaluation.bootstrap_nmi_ci(areas, [preds], n_resamples=N_BOOTSTRAP, seed=SEED)
    for name, preds in area_preds.items()
}
area_scores = pd.DataFrame(
    {
        name: {
            **evaluation.external_scores(areas, preds),
            "nmi_low": area_cis[name].low,
            "nmi_high": area_cis[name].high,
            "silhouette": evaluation.internal_score(representations[name], preds, seed=SEED),
        }
        for name, preds in area_preds.items()
    }
).T
# R3 pela média das 3 sementes, com IC da NMI média.
r3_mean_ci = evaluation.bootstrap_nmi_ci(
    areas, [area_preds[name] for name in r3_names], n_resamples=N_BOOTSTRAP, seed=SEED
)
area_scores.loc["R3-mean"] = area_scores.loc[r3_names].mean()
area_scores.loc["R3-mean", ["nmi_low", "nmi_high"]] = [r3_mean_ci.low, r3_mean_ci.high]
storage.save_table(area_scores.rename_axis("representation").reset_index(), "area_scores")
area_scores

## 4. Testes de H1 e H2

- Diferença de NMI com IC de 95% por bootstrap pareado dos documentos. R3 entra pela média das sementes.
- Critério: limite inferior do IC positivo.
- R3 − R2-CLS é exploratória e não altera a decisão sobre H2. Ver `docs/decisions.md`.

In [ ]:
r3_preds = [area_preds[name] for name in r3_names]
comparisons = {
    "H1: R3 - R1": (r3_preds, [area_preds["R1"]]),
    "H1: R4 - R1": ([area_preds["R4"]], [area_preds["R1"]]),
    "H2: R3 - R2": (r3_preds, [area_preds["R2"]]),
    "Exploratory: R3 - R2-CLS": (r3_preds, [area_preds["R2-CLS"]]),
}
differences = {
    label: evaluation.bootstrap_nmi_difference(
        areas, preds_a, preds_b, n_resamples=N_BOOTSTRAP, seed=SEED
    )
    for label, (preds_a, preds_b) in comparisons.items()
}
hypothesis_tests = pd.DataFrame(
    {
        label: {
            "difference": ci.estimate,
            "low": ci.low,
            "high": ci.high,
            "positive_lower_bound": ci.low > 0,
        }
        for label, ci in differences.items()
    }
).T
storage.save_table(hypothesis_tests.rename_axis("comparison").reset_index(), "hypothesis_tests")
hypothesis_tests

## 5. Número de grupos pela silhueta (P3)

- k de 5 a 50, sem consultar as áreas.
- O k de maior silhueta é comparado ao número de áreas.

In [ ]:
sweeps = {
    name: clustering.silhouette_sweep(x, K_RANGE, seed=SEED) for name, x in representations.items()
}
silhouette_figure = analysis.plot_silhouette_curves(sweeps)
storage.save_figure(silhouette_figure, "silhouette_by_k")

In [ ]:
silhouette_table = (
    pd.concat(sweeps, names=["representation", None])
    .reset_index(level="representation")
    .reset_index(drop=True)
)
best_rows = silhouette_table.groupby("representation", sort=False)["silhouette"].idxmax()
best_k = (
    silhouette_table.loc[best_rows, ["representation", "k"]]
    .rename(columns={"k": "best_k"})
    .assign(n_areas=n_areas)
    .reset_index(drop=True)
)
storage.save_table(silhouette_table, "silhouette_by_k")
storage.save_table(best_k, "best_k")
best_k

## 6. Análise secundária por tema

- Mantém apenas os temas com ao menos 20 documentos, antes do k-means.
- k igual ao número de temas mantidos. Mesmo bootstrap da análise por área.

In [ ]:
topic_mask = evaluation.filter_min_size(topics, min_size=MIN_TOPIC_SIZE)
kept_topics = topics[topic_mask]
n_topics = documents.loc[topic_mask, TOPIC_COLUMN].nunique()
topic_preds = {
    name: clustering.run_kmeans(x[topic_mask], k=n_topics, seed=SEED)
    for name, x in representations.items()
}
n_topics

In [ ]:
topic_cis = {
    name: evaluation.bootstrap_nmi_ci(kept_topics, [preds], n_resamples=N_BOOTSTRAP, seed=SEED)
    for name, preds in topic_preds.items()
}
topic_scores = pd.DataFrame(
    {
        name: {
            **evaluation.external_scores(kept_topics, preds),
            "nmi_low": topic_cis[name].low,
            "nmi_high": topic_cis[name].high,
            "silhouette": evaluation.internal_score(
                representations[name][topic_mask], preds, seed=SEED
            ),
        }
        for name, preds in topic_preds.items()
    }
).T
storage.save_table(topic_scores.rename_axis("representation").reset_index(), "topic_scores")
topic_scores

## 7. Projeções UMAP

- Projeção em 2 dimensões de cada representação, colorida por área.
- A mesma projeção alimenta o HDBSCAN.

In [ ]:
projections = {name: clustering.project_umap(x, seed=SEED) for name, x in representations.items()}
umap_figure = analysis.plot_umap_grid(projections, colors=areas)
storage.save_figure(umap_figure, "umap_by_area")

## 8. Termos característicos

- Termos TF-IDF de cada grupo de R3 (primeira semente).
- Composição de cada grupo por área, para comparar os termos com as áreas.

In [ ]:
reference_preds = area_preds[r3_names[0]]
top_terms = analysis.top_terms_per_cluster(texts, reference_preds, n_terms=10)
top_terms_table = pd.DataFrame(
    {"group": list(top_terms), "terms": [", ".join(terms) for terms in top_terms.values()]}
)
storage.save_table(top_terms_table, "top_terms")
top_terms_table

In [ ]:
group_area_crosstab = pd.crosstab(reference_preds, areas, rownames=["group"], colnames=["area"])
storage.save_table(group_area_crosstab.reset_index(), "group_area_crosstab")
group_area_crosstab

## 9. HDBSCAN (opcional)

- HDBSCAN sobre a projeção UMAP. O número de grupos não é fixado.
- Primeiro item da ordem de corte, em caso de atraso no cronograma.

In [ ]:
# Mesmo tamanho mínimo adotado na análise por tema.
hdbscan_preds = {
    name: clustering.run_hdbscan(x_2d, min_cluster_size=MIN_TOPIC_SIZE)
    for name, x_2d in projections.items()
}
hdbscan_scores = pd.DataFrame(
    {
        name: {
            **evaluation.external_scores(areas, preds),
            # O rótulo -1 indica ruído e não conta como grupo.
            "n_groups": len(set(preds) - {-1}),
        }
        for name, preds in hdbscan_preds.items()
    }
).T
storage.save_table(hdbscan_scores.rename_axis("representation").reset_index(), "hdbscan_scores")
hdbscan_scores